# Summary of lecture 2

Yesterday we introduced interval arithmetic, an arithmetic in which every operation returns an interval that is guaranteed to contain the true mathematical result; the property that makes it useful is range enclosure, i.e. the evaluation of a function on an interval $I$ returns an interval containing $f(I)$.

On top of it we built three tools:
1. the interval Newton method, which proves that a root exists in an interval and, when the Newton image is strictly contained in the interval, that it is unique;
2. Taylor models, which approximate a function on an interval by a polynomial together with an interval $\Delta$ containing the error;
3. rigorous integration, which uses the Taylor model of the integrand on each subinterval to enclose the value of an integral.

We closed by saying that these are the engine of the Ulam method: the interval Newton method computes the preimages that give the entries of the discretized transfer operator, so that the computed matrix is a matrix of intervals containing the entries of the true one, and rigorous integration turns an approximation of the invariant density $h$ into rigorous Birkhoff averages, through the identity
$$
\lim_{n\to +\infty}\frac{1}{n}\sum_{i=0}^{n-1} \phi(T^i(x)) = \int \phi\, h\, dx.
$$

Today we run that programme to the end, on a random dynamical system rather than on a deterministic one, and we prove a theorem with it: for one family of unimodal maps the Lyapunov exponent is positive when the noise is small and negative when the noise is large, so the family undergoes a transition.

# Stationary measures of random dynamical systems with additive uniform noise

We study the random dynamical system
$$
X_{n+1} = \pi\big(T(X_n) + \Omega_n\big),
$$
where the $\Omega_n$ are independent and identically distributed random variables with density $\rho_\xi$, uniform on $[-\xi, \xi]$, and $\pi$ is a boundary condition, either the identification of $0$ with $1$, which we call periodic, or the reflection at the two endpoints, which we call reflecting.

The measure of a set after one step is the average, over the noise, of the measures of its preimages; the operator that carries this out on densities is the annealed transfer operator
$$
L_\xi f = \rho_\xi * (L f),
$$
where $L$ is the transfer operator of the deterministic map and $*$ is convolution.
Its fixed point is the stationary density of the random system, and it is the object we are going to enclose.

The technique is the one used in [Galatolo, Monge, Nisoli, *Existence of noise induced order, a computer aided proof*](https://iopscience.iop.org/article/10.1088/1361-6544/ab86cd) and in [Chihara, Sato, Nisoli, Galatolo, *Existence of multiple noise-induced transitions in Lasota-Mackey maps*](https://pubs.aip.org/aip/cha/article/32/1/013117/2835597/Existence-of-multiple-noise-induced-transitions-in); those papers needed accelerated computations on a graphics card, and what we do today is a toy version of the same steps, small enough to run in a lecture.

## Setting up

In [ ]:
import Pkg
Pkg.activate("./")
Pkg.add(["RigorousInvariantMeasures", "BallArithmetic", "IntervalArithmetic",
         "Plots", "RecipesBase", "LaTeXStrings"])

In [ ]:
using RigorousInvariantMeasures, IntervalArithmetic, BallArithmetic
using Plots, RecipesBase, LaTeXStrings

Three packages export a function called `mid`, and two of them export `inf`, `sup` and `radius`; loading them together leaves those names ambiguous, and Julia refuses to guess.
We resolve the ambiguity once, by importing explicitly the versions we mean, and we take from the package the two internal names we shall need, `UniformKernelUlam` and `gamma`, together with `Observable`.
The plotting recipes for the Ulam basis live in an extension that is loaded only when `Plots`, `RecipesBase` and `LaTeXStrings` are all present, which is why the three of them appear above.

In [ ]:
import IntervalArithmetic: inf, sup, mid, radius, diam, interval
using RigorousInvariantMeasures: UniformKernelUlam, gamma, Observable

setdisplay(:infsup; decorations = false, ng_flag = false)

## Why the random case is the easier entry point

For a deterministic map the whole method rests on a Lasota-Yorke inequality
$$
\|L^n f\|_{BV} \leq A\lambda^n \|f\|_{BV} + B\|f\|_{L^1},
$$
which has to be established for the map at hand, with explicit constants; the term $A\lambda^n\|f\|_{BV}$ is what forces the strong norm to appear everywhere in the estimates that follow.

For additive noise the corresponding inequality is one line, and it has no strong norm on the right hand side: since $L_\xi f = \rho_\xi * (Lf)$ and $L$ is a weak contraction in $L^1$,
$$
\|L_\xi f\|_{BV} \leq \|\rho_\xi\|_{BV}\,\|f\|_{L^1} = \frac{1}{\xi}\|f\|_{L^1}.
$$
The package returns the pair $(A, B)$ of that inequality for a noise kernel, and $A$ is $0$; there is no term to fight, and no computation to do on the map.

We work on the Ulam basis, the partition of $[0,1]$ into $k$ intervals of equal length, on which a density is represented by its average on each interval.

In [ ]:
k = 1024
B = Ulam(k)

## The noise kernel in the Ulam basis

The package carries two discretizations of the uniform kernel, and they are two different operators, not a fast and a slow version of the same one.

`UniformNoiseUlam(ξ, B)` builds a window of $2\lceil \xi k\rceil$ cells, an even number, with the two end weights reduced by a fraction so that the total weight is exactly the one of the density $\rho_\xi$; it applies the window by a direct sum over the cells.

`UniformKernelUlamPeriodic(B, l)` and `UniformKernelUlamReflecting(B, l)` build a window of $2l+1$ cells, an odd number, all of weight $1/(2l+1)$, and apply it by a sliding sum: the sum over the window at cell $i+1$ is obtained from the one at cell $i$ by adding one term and subtracting another, with the subtraction error tracked by compensated summation. The cost per cell is then independent of the width of the window.

No choice of $l$ makes the second reproduce the first, since one window is even and the other odd.

In [ ]:
NK_old = UniformNoiseUlam(0.05, B)
length(NK_old.v), NK_old.v[1], NK_old.v[2]

In [ ]:
NK_new = UniformKernelUlamPeriodic(B, 51)
2*NK_new.l+1

On the same density the two differ by a few parts in a thousand at $k=1024$, and the difference shrinks as the partition is refined; on this partition the sliding sum is about two orders of magnitude faster, and the gain grows with the width of the window, since the direct sum pays for every cell of the window and the sliding sum does not.

In [ ]:
v = [exp(-(((i-0.5)/k - 0.3)^2)/0.01) for i in 1:k]
a = NK_old*copy(v)
b = NK_new*copy(v)
maximum(abs.(a .- b))/maximum(abs.(a))

In [ ]:
t_old = @elapsed for _ in 1:20; NK_old*copy(v); end
t_new = @elapsed for _ in 1:20; NK_new*copy(v); end
t_old/t_new

We use the sliding sum kernel throughout this lecture; the 2025 edition of this school used `UniformNoiseUlam` instead, so the numbers below are close to, and not equal to, the ones of that notebook.
The reason for the choice is that this is the kernel of [Galatolo, Monge, Nisoli](https://iopscience.iop.org/article/10.1088/1361-6544/ab86cd); that paper uses reflecting boundary conditions, so we shall use the reflecting kernel wherever the boundary matters.

### Two defects of the released version

In version 0.3.0 of `RigorousInvariantMeasures`, which is the one in the registry as this notebook is written, the sliding sum kernel has two defects; both are repaired on the branch `fix-noise-kernels` and will land in the next release, and we show them here because the released version is what a reader installing the package today will get.

The first is in the path that applies the kernel to a vector of intervals. The floating point path is correct, and it is the one the norm estimates below use; the interval path adds a rounding term computed with the wrong constant, which makes the radius about $\|v\|_1/n$ rather than the rounding level, so the enclosure encloses nothing useful. We therefore compute the residual in ball arithmetic further down rather than through the interval path.

In [ ]:
vi = ones(Interval{Float64}, k)
radius((NK_new*copy(vi))[1]), radius((NK_old*copy(vi))[1])

The second defect is in `dfly`, and it is the one that matters for a proof, since a constant that is too small is not a safe error.

Lemma 47 of Galatolo-Monge-Nisoli gives $\|N_\xi\|_{L^1\to \mathrm{Var}} \leq \mathrm{Var}(\rho_\xi)$, and the uniform density of half-width $\xi$ has height $1/(2\xi)$ over a support of length $2\xi$, so it rises once and falls once and $\mathrm{Var}(\rho_\xi) = 1/\xi$. The released version returned $1/(2\xi)$, half of that, and moreover computed $\xi$ as $(2l+1)/k$, which is the full width of the window and not the half-width, so the constant it returned was a quarter of the correct $2k/(2l+1)$.

The bound is attained, so the value can be checked rather than argued about: a spike of unit $L^1$ mass is spread to a box of height $1/(2\xi)$, whose variation is exactly $1/\xi$. We verify this below, and we remark that the older kernel returned half the correct constant for the same reason, so the two disagreed with each other.

In [ ]:
import RigorousInvariantMeasures: dfly

# Var(ρ_ξ) = 1/ξ, with ξ = (2l+1)/(2k) for the sliding sum kernel and ξ the
# half-width for the older one. Both released methods return less than this.
# Delete this cell once a release carrying the repair is out.
dfly(::Type{TotalVariation}, ::Type{L1}, N::UniformKernelUlam) =
    (0.0, sup(interval(2*length(N.B))/interval(2*N.l+1)))

dfly(::Type{TotalVariation}, ::Type{L1}, N::RigorousInvariantMeasures.DiscretizedNoiseKernelUlam) =
    (0.0, sup(interval(1)/N.ξ))

In [ ]:
# the constant is attained by a spike, so we can check it rather than trust it
spike = zeros(k)
spike[k ÷ 2] = 1.0
attained = normbound(B, TotalVariation, NK_new*copy(spike)) / normbound(B, L1, spike)
attained, dfly(TotalVariation, L1, NK_new)[2]

In [ ]:
dfly(TotalVariation, L1, NK_new), dfly(TotalVariation, L1, NK_old)

The pair is $(A, B)$ of the inequality above: $A$ is zero, so no strong norm survives on the right hand side, and $B = 1/\xi$, matching the variation the spike attains.

### Choosing the noise size

The window of the sliding sum kernel is an odd number of cells, so the kernel represents exactly the uniform density on $[-\xi, \xi]$ only when $2\xi k$ is an odd integer; we choose the noise sizes so that this holds, rather than choosing a round decimal and discretizing it approximately.

The same has to hold on the fine partition, and $2\xi k_{\text{fine}} = (k_{\text{fine}}/k)\,2\xi k$, so the ratio between the two partitions must itself be odd; we take it to be $63$, which puts the fine partition at $64512$ cells, close to the $65536$ one would use out of habit.
For $\xi$ near $0.05$ this gives a window of $103$ coarse cells and $\xi = 103/2048$.

In [ ]:
kf = 63*1024
Bf = Ulam(kf)
RigorousInvariantMeasures.is_refinement(Bf, B)

## The two black boxes

Two functions of the package do the work of the estimate, and we use them here without deriving them.

`powernormboundsnoise(B; Q, NK)` returns a vector `norms` such that `norms[n]` is a rigorous upper bound for $\|(N_h Q_h)^n\|_{L^1}$ restricted to the space of vectors of average zero, where $Q_h$ is the discretized transfer operator and $N_h$ the discretized kernel. It computes the bound column by column, applying the operator to each of the $k-1$ vectors $e_1 - e_{j+1}$ that span that space and accumulating the floating point error committed on the way; the cost is quadratic in $k$, so it is affordable only on a coarse partition.

`finepowernormboundsnoise(B, Bfine, norms; Qfine, NKfine)` transports those bounds to a finer partition. It combines the coarse bounds with the projection error between the two partitions, which for the Ulam basis is $1/(2k)$, and with the trivial bounds coming from $\|N_h Q_h\| \leq 1$; the cost is linear in the size of the fine matrix, since no column sweep is performed there. Computing the norms where the matrix is small and transporting them to where the matrix is large is what makes the fine estimate possible at all.

Both are declared black boxes for today; what we need from them is the contract, that `norms[n]` bounds the norm of the $n$-th power on the average zero space.

## The residual, and where ball arithmetic comes in

Once we have an approximate fixed point $w$ of the discretized operator, the distance between $w$ and the true fixed point is controlled by the residual $\|N_h Q_h w - w\|_{L^1}$ together with the norms of the powers; the package computes the residual in `residualboundnoise`, which multiplies an interval vector by the kernel and so runs into the defective interval path described above.

We compute the residual instead in ball arithmetic, where a vector is a centre and a radius rather than a pair of endpoints, and the radius is propagated explicitly. The centre is the floating point application of the kernel, which is correct; the radius is the input radius transported by an operator of norm one, plus the rounding error of the $n$ additions of the window and of the final division, both divided by $n$. We remark that the same could be done for the whole a posteriori estimate, avoiding interval linear algebra entirely.

In [ ]:
function residual_ball(B, Q, NK, w)
    u  = Q*w                                # sparse interval matrix times a float vector
    uc = IntervalArithmetic.mid.(u)
    ur = IntervalArithmetic.radius.(u)
    n  = 2*NK.l+1
    kk = length(w)
    c  = NK*copy(uc)                        # the floating point path of the kernel
    γ  = gamma(Float64, n+1)
    ϵ  = setrounding(Float64, RoundUp) do
        (γ*sum(abs, uc) + sum(ur))/n
    end
    r = BallVector(c, fill(ϵ, kk)) - BallVector(w)
    return setrounding(Float64, RoundUp) do
        (sum(abs, r.c) + sum(r.r))/kk
    end
end

# A first example

We take the expanding circle map $T(x) = 2x + \frac{1}{2}x(1-x) \bmod 1$ with noise of size $\xi = 103/2048$ and periodic boundary conditions, which are the natural ones on the circle.

In [ ]:
D = mod1_dynamic(x -> 2*x + 0.5*x*(1-x))

In [ ]:
Q  = DiscretizedOperator(B,  D)
Qf = DiscretizedOperator(Bf, D)

In [ ]:
NK  = UniformKernelUlamPeriodic(B,  51)
NKf = UniformKernelUlamPeriodic(Bf, 63*51+31)
(2*NK.l+1, 2*NKf.l+1, 63*(2*NK.l+1))

The two windows are the same noise, seen on the two partitions; the fine window is $63$ times the coarse one, cell for cell.

In [ ]:
@time norms = powernormboundsnoise(B; Q = Q, NK = NK)

In [ ]:
plot(norms; yscale = :log10, label = "norms of the powers on the average zero space")

The norms start at $1$, since the operator preserves the integral, and then decay; the decay is what the a posteriori estimate turns into a bound on the distance from the fixed point.

In [ ]:
@time w = invariant_vector_noise(B, Q, NK; iter = 40)

In [ ]:
ε₁ = residual_ball(B, Q, NK, w)
err = distance_from_invariant_noise(B, Q, NK, w, norms; ε₁ = ε₁)

The number `err` is an upper bound, in $L^1$, for the distance between the piecewise constant density drawn below and the stationary density of the random system; the second curve in the plot is that bound.

In [ ]:
plot(B, w)
plot!(B, err)

On the coarse partition the bound is of the order of a tenth, which is not much of a theorem. We now transport the norms to the fine partition and repeat.

In [ ]:
@time fine_norms = finepowernormboundsnoise(B, Bf, norms; Qfine = Qf, NKfine = NKf)

In [ ]:
@time wf = invariant_vector_noise(Bf, Qf, NKf; iter = 40)
ε₁f = residual_ball(Bf, Qf, NKf, wf)
errf = distance_from_invariant_noise(Bf, Qf, NKf, wf, fine_norms; ε₁ = ε₁f)

In [ ]:
plot(Bf, wf)
plot!(Bf, errf)

The bound has dropped by about a factor of fifty, which is what the projection error $1/(2k)$ predicts for a refinement by $63$.

When the noise is additive we can also bound the norms of the powers of the operator that acts on measures, and not only of its discretization; the bound is obtained from the coarse norms by the same coarse to fine argument with the fine partition replaced by the space itself.

In [ ]:
abstract_norms = abstractpowernormboundsnoise(B, NK, norms)
plot(abstract_norms; yscale = :log10, label = "norms of the powers of the annealed operator")

# A transition of the Lyapunov exponent

We turn to the family studied in [Nisoli, *How does noise induce order*](https://arxiv.org/abs/2003.08422),
$$
T_{\alpha,\beta}(x) = 2\beta|x|^{\alpha} - 1,
$$
on $[-1,1]$; we change coordinates so that the system lives on $[0,1]$. As $\alpha$ grows the part of the space on which the map contracts grows with it, and adding noise makes the orbit visit that part more often, so the Lyapunov exponent can fall below zero as the noise size increases.

The argument in that paper is analytic and proves the transition for large noise, by showing that the stationary measure is close in $BV$ to the uniform measure; we prove the transition here by computing both signs, at $\beta = 1$ and $\alpha = 4$, with the noise sizes $\xi = 103/2048$ and $\xi = 819/2048$.

In [ ]:
D2 = PwMap([x -> 1-2^4*(0.5-x)^4, x -> 1-2^4*(x-0.5)^4],
           [interval(0), interval(0.5), interval(1)],
           [0 1;
            1 0])

The plotting recipe for a `PwMap` compares an ordinary number with an interval, which version 1 of `IntervalArithmetic` refuses to decide, so we draw the two branches as a plain function.

In [ ]:
T₂(x) = x <= 0.5 ? 1-2^4*(0.5-x)^4 : 1-2^4*(x-0.5)^4
plot(T₂, 0, 1; label = "T at beta = 1, alpha = 4")

In [ ]:
Q2  = DiscretizedOperator(B,  D2)
Q2f = DiscretizedOperator(Bf, D2)

## The logarithm of the derivative

The Lyapunov exponent of the random system is
$$
\lambda = \int \log|T'|\, f\, dx,
$$
with $f$ the stationary density, so we need the observable $H = \log|T'|$ discretized on the Ulam basis, i.e. the vector whose $i$-th entry is $k$ times the integral of $H$ on the $i$-th cell.

The generic routine of the package integrates the observable with Taylor models and fails here, since $T'$ vanishes at the critical point and $H$ is unbounded there. For this family we do not need it: the derivative is
$$
T'(x) = -2^{\alpha}\alpha(1/2-x)^{\alpha-1} \quad \text{on } [0,1/2],
$$
symmetric on the other half, and a primitive of $H$ is
$$
\int \log\big(2^{\alpha}\alpha(1/2-x)^{\alpha-1}\big)\,dx = (x-1/2)\log\big(2^{\alpha}\alpha(1/2-x)^{\alpha-1}\big) - \alpha x + x,
$$
so each cell integral is a difference of two evaluations, and we use the symmetry to halve the work. The two central cells are the ones on which the primitive is singular, and their value is the limit, which we write out by hand since the computer cannot take it.

In [ ]:
α = interval(4)

H(x)     = log(2^α*α*(interval(0.5)-x)^(α-1))
Hprim(x) = (x-interval(0.5))*H(x) - α*x + x

In [ ]:
function logder_unimodal(B, α)
    N = length(B)
    @assert iseven(N)
    v = zeros(Interval{Float64}, N)
    for i in 1:N÷2-1
        v[i] = Hprim(interval(B.p[i+1])) - Hprim(interval(B.p[i]))
        v[end-i+1] = v[i]
    end
    v[N÷2]   = -α*interval(0.5) + interval(0.5) - Hprim(interval(B.p[N÷2]))
    v[N÷2+1] = v[N÷2]
    v *= interval(N)
    return Observable(B, v, interval(-Inf, Inf))
end

In [ ]:
logder = logder_unimodal(Bf, α)
logder.v[1], logder.v[kf÷2]

In [ ]:
plot(Bf, logder.v; ylims = (-NaN, NaN), label = "log|T'| on the Ulam basis")

The third argument of `Observable` is a bound on the $L^\infty$ norm of the observable, which the package uses to turn an $L^1$ bound on the density into a bound on the integral; here that norm is infinite, and the generic pairing would return an infinite enclosure. The way out is to split the space.

## Splitting the space

We use Corollary 30 of [Galatolo, Monge, Nisoli](https://iopscience.iop.org/article/10.1088/1361-6544/ab86cd). Its hypotheses are that $f$ is the stationary density, $f_k$ its computed approximation, and $E$ any measurable subset of $X = [0,1]$; its conclusion is
$$
\Big|\int H\,df - \int H\,df_k\Big| \leq \|H\|_{L^1(E)}\|f\|_{L^{\infty}(E)}
 + \frac{\big|\sup_{X\setminus E} H + \inf_{X\setminus E} H\big|}{2}\,\|f - f_k\|_{L^1}.
$$
On $E$, a neighbourhood of the critical point, the observable is unbounded but integrable, and we pay $\|f\|_{L^\infty}$; off $E$ the observable is bounded, and we pay the $L^1$ error on the density, multiplied by the mean of its extreme values rather than by their maximum, which is what makes the bound useful when $H$ changes sign.

The $L^\infty$ norm of the stationary density is bounded by the same constant $1/\xi$ that appears in the inequality of the first section, since $f = L_\xi f = \rho_\xi * (Lf)$ and $\|Lf\|_{L^1} = 1$; the code reads it off `dfly`.

There is a choice to make, the size of $E$; we take it to be a symmetric union of cells around the critical point and we try the first ten sizes.

In [ ]:
function corollary30(B, NK, logder, error_fine; kmax = 10)
    out = zeros(Float64, kmax)
    N = length(B) ÷ 2
    S = H(interval(0))                       # the value at the far end of E
    _, C = dfly(TotalVariation, L1, NK)      # the bound on ||f||_∞
    for i in 0:kmax-1
        m = H(interval(B.p[N-i]))
        out[i+1] = sup(abs(m+S)/2*interval(error_fine)
                       + 2*interval(C)*sum(abs.(logder.v)[N-i:N])/interval(length(B)))
    end
    return out
end

## Small noise

In [ ]:
NK_s  = UniformKernelUlamReflecting(B,  51)
NK_sf = UniformKernelUlamReflecting(Bf, 63*51+31)
(2*NK_s.l+1)/(2*k)

In [ ]:
@time norms_s = powernormboundsnoise(B; Q = Q2, NK = NK_s)
@time fine_s  = finepowernormboundsnoise(B, Bf, norms_s; Qfine = Q2f, NKfine = NK_sf)
fine_s[1:8]

In [ ]:
w_s = invariant_vector_noise(Bf, Q2f, NK_sf; iter = 40)
e_s = residual_ball(Bf, Q2f, NK_sf, w_s)
err_s = distance_from_invariant_noise(Bf, Q2f, NK_sf, w_s, fine_s; ε₁ = e_s)

In [ ]:
plot(Bf, w_s)
plot!(Bf, err_s)

In [ ]:
val_s = (logder.v'*w_s)/interval(kf)

In [ ]:
ve_s = corollary30(Bf, NK_sf, logder, err_s)

Of the ten sizes the first is the best, so the smallest $E$ wins here; the enclosure of the Lyapunov exponent is then the following.

In [ ]:
λ_small = val_s + interval(-ve_s[1], ve_s[1])

The interval is wide, and it lies entirely to the right of zero: at noise size $103/2048$ the Lyapunov exponent of this random system is positive.

## Large noise

We repeat with $\xi = 819/2048$, which is close to $0.4$ and again gives an odd window; the discretized transfer operators `Q2` and `Q2f` do not depend on the noise, so they are not recomputed.

In [ ]:
NK_l  = UniformKernelUlamReflecting(B,  409)
NK_lf = UniformKernelUlamReflecting(Bf, 63*409+31)
(2*NK_l.l+1)/(2*k)

In [ ]:
@time norms_l = powernormboundsnoise(B; Q = Q2, NK = NK_l)
@time fine_l  = finepowernormboundsnoise(B, Bf, norms_l; Qfine = Q2f, NKfine = NK_lf)
fine_l[1:8]

In [ ]:
w_l = invariant_vector_noise(Bf, Q2f, NK_lf; iter = 40)
e_l = residual_ball(Bf, Q2f, NK_lf, w_l)
err_l = distance_from_invariant_noise(Bf, Q2f, NK_lf, w_l, fine_l; ε₁ = e_l)

In [ ]:
plot(Bf, w_l)
plot!(Bf, err_l)

In [ ]:
val_l = (logder.v'*w_l)/interval(kf)
ve_l = corollary30(Bf, NK_lf, logder, err_l)
λ_large = val_l + interval(-ve_l[1], ve_l[1])

The enclosure lies entirely to the left of zero: at noise size $819/2048$ the Lyapunov exponent is negative.

Together with the previous computation this is a computer assisted proof that the Lyapunov exponent of the family at $\beta = 1$, $\alpha = 4$ changes sign as the noise size grows; the two computations are independent, and each of them is a chain of enclosures whose every link we have either derived or declared.

# Summary of the lecture

We approximated the stationary density of a random dynamical system with additive uniform noise, with a rigorous bound on the $L^1$ distance from the true one, and we used it to enclose the Lyapunov exponent.

The pieces were:
1. the inequality $\|L_\xi f\|_{BV}\leq \frac{1}{\xi}\|f\|_{L^1}$, which replaces the Lasota-Yorke inequality of the deterministic case and costs one line;
2. the Ulam discretization of the map and of the noise kernel, the second by a sliding sum whose cost per cell does not depend on the width of the window;
3. `powernormboundsnoise` and `finepowernormboundsnoise`, which compute the norms of the powers where the matrix is small and transport them to where it is large;
4. an a posteriori estimate from the residual and those norms, with the residual computed in ball arithmetic;
5. Corollary 30, which integrates an unbounded observable against the density by splitting the space.

Applied twice, at two noise sizes, this proves a transition of the Lyapunov exponent in the family of [Nisoli, *How does noise induce order*](https://arxiv.org/abs/2003.08422).

Tomorrow we replace the Ulam basis by a Fourier basis, which is the right one when the noise is smooth, and we ask for a certified mixing rate rather than for a stationary density.